In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from astropy.wcs import WCS
from reproject import reproject_to_healpix

In [ ]:
def read_in_file(file_in):

    hdu  = fits.open(file_in)
    hdr  = hdu[0].header
    data = hdu[0].data
    
    return data, hdr

In [ ]:
Q_C, hdrQ_C = read_in_file('/srv/data/chime/chime_QU_Oct2023_400_729/Q_400_729_Oct2023_new_eq.hpx.fits')
U_C, hdrU_C = read_in_file('/srv/data/chime/chime_IQUV_Mar2024_400_729/U_400_729_Mar2024_new.hpx.fits')
I_C, hdrI_C = read_in_file('/srv/data/chime/chime_QU_Oct2023_400_729/I_400_729_Oct2023_new.hpx.fits')


In [ ]:
Q_D, hdrQ_D = read_in_file('/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/dragons_freqK_eq_q.hpx.fits')
U_D, hdrU_D = read_in_file('/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/dragons_freqK_eq_u.hpx.fits')
I_D, hdrI_D = read_in_file('/srv/data/dragons/dragons-canfar-Nov2025/frequency_cubes/dragons_freqK_eq_i.hpx.fits')


In [ ]:
freqs_C = WCS(hdrQ_C).all_pix2world(0,range(Q_C.shape[0]),0)[1]
freqs_D = WCS(hdrQ_D).all_pix2world(0,range(Q_D.shape[0]),0)[1]


In [ ]:
nside_C = hp.npix2nside(Q_C.shape[1])
nside_D = hp.npix2nside(Q_D.shape[1])
print(f'CHIME nside={nside_C}')
print(f'DRAGONS nside={nside_D}')

In [ ]:
def map_freq_slice(fpick):

    # Find frequency indices to use
    idx_D = abs(fpick - freqs_D).argmin()
    idx_C = abs(freqs_D[idx_D] - freqs_C).argmin()

    print(freqs_C[idx_C])
    print(freqs_D[idx_D])

    # Make matched sampling maps
    Q_C_use = hp.ud_grade(Q_C[idx_C], nside_D)
    U_C_use = hp.ud_grade(U_C[idx_C], nside_D)
    I_C_use = hp.ud_grade(I_C[idx_C], nside_D)
    
    Q_D_use = Q_D[idx_D]
    U_D_use = U_D[idx_D]
    I_D_use = I_D[idx_D]

    # Find max values to plot for Stoke Q, U, I
    PI_C = np.sqrt(Q_C_use**2 + U_C_use**2)
    PI_D = np.sqrt(Q_D_use**2 + U_D_use**2)

    PI_C[PI_C > 1e10] = np.nan
    PI_D[PI_D > 1e10] = np.nan

    I_C_use[abs(I_C_use) > 1e10] = np.nan
    I_D_use[abs(I_D_use) > 1e10] = np.nan

    PImax_C = np.nanmedian(PI_C)*5
    PImax_D = np.nanmedian(PI_D)*5

    Imin_C = np.nanmin(I_C_use)
    Imax_C = np.nanmedian(I_C_use - Imin_C)*5
    Imax_D = np.nanmedian(I_D_use)*5
    print(Imax_C, Imax_D)

    # Make frequency labels
    fC = np.round(freqs_C[idx_C]/1e6,3)
    fD = np.round(freqs_D[idx_D]/1e6,3)

    # Make ratios
    Q_rat = Q_C_use/Q_D_use
    U_rat = U_C_use/U_D_use
    I_rat = (I_C_use-Imin_C)/I_D_use

    Q_rat[abs(Q_C_use)>1e10] = np.nan 
    U_rat[abs(U_C_use)>1e10] = np.nan 
    I_rat[abs(I_C_use)>1e10] = np.nan 

    # Full resolution Q, U, I maps
    hp.mollview(Q_C[idx_C], min=-PImax_C, max=PImax_C, cmap='RdBu_r', title=f'CHIME Q, {fC} MHz', unit='Jy/beam')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/Q_maps/Q_{fC}.png')
    plt.close()
    hp.mollview(U_C[idx_C], min=-PImax_C, max=PImax_C, cmap='RdBu_r', title=f'CHIME U, {fC} MHz', unit='Jy/beam')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/U_maps/U_{fC}.png')
    plt.close()
    hp.mollview(I_C[idx_C]-Imin_C, min=0, max=Imax_C, cmap='cubehelix', title=f'CHIME I, {fC} MHz', unit='Jy/beam')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/I_maps/I_{fC}.png')
    plt.close()
    hp.mollview(Q_D[idx_D], min=-PImax_D, max=PImax_D, cmap='RdBu_r', title=f'DRAGONS Q, {fD} MHz', unit='K')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/Q_maps/Q_{fD}.png')
    plt.close()
    hp.mollview(U_D[idx_D], min=-PImax_D, max=PImax_D, cmap='RdBu_r', title=f'DRAGONS U, {fD} MHz', unit='K')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/U_maps/U_{fD}.png')
    plt.close()
    hp.mollview(I_D[idx_D], min=0,        max=Imax_D, cmap='cubehelix', title=f'DRAGONS I, {fD} MHz', unit='K')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/I_maps/I_{fD}.png')
    plt.close()

    # Matched resolution ratio maps
    hp.mollview(Q_rat, min=-4, max=4, cmap='RdBu_r', title=f'Q ratio, DRAGONS {fD}, CHIME {fC} MHz', unit='Jy/beam/K')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/Q_ratio_maps/Q_{fC}.png')
    plt.close()
    hp.mollview(U_rat, min=-4, max=4, cmap='RdBu_r', title=f'U ratio, DRAGONS {fD}, CHIME {fC} MHz', unit='Jy/beam/K')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/U_ratio_maps/U_{fC}.png')
    plt.close()
    hp.mollview(I_rat, min=0, max=4, cmap='cubehelix', title=f'I ratio, DRAGONS {fD}, CHIME {fC} MHz', unit='Jy/beam/K')
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/I_ratio_maps/I_{fC}.png')
    plt.close()

    fig, ax = plt.subplots(1,3,figsize=(12,3))
    plt.subplots_adjust(left=0.05, bottom=0.2, right=0.99, top=0.9, hspace=0.1)
    fs = 14
    ax[0].hist(Q_rat, color='C0', bins=100, range=(-6,6));
    ax[1].hist(U_rat, color='C1', bins=100, range=(-6,6));
    ax[2].hist(I_rat, color='C2', bins=100, range=(0,6));
    ax[0].set_xlim(-6,6)
    ax[1].set_xlim(-6,6)
    ax[2].set_xlim(0,6)
    fig.suptitle(f'Distributions of ratios, DRAGONS {fD}, CHIME {fC} MHz', fontsize=fs)
    ax[0].set_xlabel(r'$Q_{CHIME}/Q_{DRAGONS}$')
    ax[1].set_xlabel(r'$U_{CHIME}/U_{DRAGONS}$')
    ax[2].set_xlabel(r'$I_{CHIME}/I_{DRAGONS}$')
    for i in range(0,3):
        ax[i].grid(True)
    plt.savefig(f'/home/aordog/Dropbox/plots/chime/dragons_chime_eq_May2026/ratio_hist/hist_{fC}.png')
    plt.close()

    return 


In [ ]:
map_freq_slice(410e6)


In [ ]:
fpick_list = freqs_D[np.where((freqs_D >= 400e6) & (freqs_D <= 729e6))]
for j in range(0,len(fpick_list)):
    try:
        map_freq_slice(fpick_list[j])
    except:
        pass